In [3]:
%cd /Users/andrinschumacher/Downloads/notebook

/Users/andrinschumacher/Downloads/notebook


## Quality control

In [4]:
!mkdir -p ./output2/fastp

!fastp \
  -i ./data/FASTQ_Chr22/S01_1.fq.gz \
  -I ./data/FASTQ_Chr22/S01_2.fq.gz \
  -o ./output2/fastp/clean_S01_1.fq.gz \
  -O ./output2/fastp/clean_S01_2.fq.gz \
  -q 30 \
  -h ./output2/fastp/S01_fastp_report.html

Read1 before filtering:
total reads: 533629
total bases: 80044350
Q20 bases: 78702480(98.3236%)
Q30 bases: 75774426(94.6656%)
Q40 bases: 127238(0.158959%)

Read2 before filtering:
total reads: 533629
total bases: 80044350
Q20 bases: 77176585(96.4173%)
Q30 bases: 71508958(89.3367%)
Q40 bases: 188829(0.235905%)

Read1 after filtering:
total reads: 504415
total bases: 75346373
Q20 bases: 74240776(98.5326%)
Q30 bases: 71761817(95.2426%)
Q40 bases: 122550(0.162649%)

Read2 after filtering:
total reads: 504415
total bases: 75346373
Q20 bases: 73313535(97.302%)
Q30 bases: 68889368(91.4302%)
Q40 bases: 186565(0.24761%)

Filtering result:
reads passed filter: 1008830
reads failed due to low quality: 58428
reads failed due to too many N: 0
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 15290
bases trimmed due to adapters: 659908

Duplication rate: 0.0457247%

Insert size peak (evaluated by paired-end reads): 269

JSON report: fastp.json
HTML rep

## Index reference genome

In [5]:
!bwa index ./reference_chr22/hg38_GRCh38.chr22.fa

[bwa_index] Pack FASTA... 0.21 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=101636936, availableWord=19151484
[BWTIncConstructFromPacked] 10 iterations done. 31590664 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 58359704 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 82148056 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 101636936 characters processed.
[bwt_gen] Finished constructing BWT in 40 iterations.
[bwa_index] 13.75 seconds elapse.
[bwa_index] Update BWT... 0.16 sec
[bwa_index] Pack forward-only FASTA... 0.13 sec
[bwa_index] Construct SA from BWT and Occ... 6.99 sec
[main] Version: 0.7.19-r1273
[main] CMD: bwa index ./reference_chr22/hg38_GRCh38.chr22.fa
[main] Real time: 21.297 sec; CPU: 21.369 sec


## Alignment with bwa

In [6]:
!mkdir -p ./output2/bwa

!bwa mem -M -t 2 \
  -R "@RG\tID:S01\tPL:ILLUMINA\tLB:lib1\tSM:S01" \
  ./reference_chr22/hg38_GRCh38.chr22.fa \
  ./output2/fastp/clean_S01_1.fq.gz \
  ./output2/fastp/clean_S01_2.fq.gz \
  > ./output2/bwa/S01.sam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 133866 sequences (20000070 bp)...
[M::process] read 133906 sequences (20000148 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (2, 53310, 2, 0)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (299, 353, 403)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (91, 611)
[M::mem_pestat] mean and std.dev: (350.18, 80.15)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 715)
[M::mem_pestat] skip orientation RF as there are not enough pairs
[M::mem_pestat] skip orientation RR as there are not enough pairs
[M::mem_process_seqs] Processed 133866 reads in 12.182 CPU sec, 6.104 real sec
[M::process] read 133916 sequences (20000224 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (1, 53261, 4, 1)
[M::mem_pestat] skip orientatio

## Convert SAM to BAM

In [7]:
!samtools view -@ 2 -bS \
  ./output2/bwa/S01.sam \
  > ./output2/bwa/S01.bam

## Sort BAM

In [8]:
!samtools sort -@ 2 \
  -o ./output2/bwa/S01_sorted.bam \
  ./output2/bwa/S01.bam

[bam_sort_core] merging from 0 files and 2 in-memory blocks...


## Index BAM

In [9]:
!samtools index -@ 2 ./output2/bwa/S01_sorted.bam

## Filter mapped reads only

In [10]:
!samtools view -b \
  -F 4 -F 8 \
  -@ 2 \
  ./output2/bwa/S01_sorted.bam \
  > ./output2/bwa/S01_sorted_mapped.bam

## Stats after filtering

In [11]:
!samtools flagstat ./output2/bwa/S01_sorted_mapped.bam

1010218 + 0 in total (QC-passed reads + QC-failed reads)
1008746 + 0 primary
1472 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
1010218 + 0 mapped (100.00% : N/A)
1008746 + 0 primary mapped (100.00% : N/A)
1008746 + 0 paired in sequencing
504373 + 0 read1
504373 + 0 read2
1007312 + 0 properly paired (99.86% : N/A)
1008746 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


## Reference variant calling

In [12]:
!samtools faidx ./reference_chr22/hg38_GRCh38.chr22.fa

In [13]:
!gatk CreateSequenceDictionary \
  -R ./reference_chr22/hg38_GRCh38.chr22.fa \
  -O ./reference_chr22/hg38_GRCh38.chr22.dict

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar CreateSequenceDictionary -R ./reference_chr22/hg38_GRCh38.chr22.fa -O ./reference_chr22/hg38_GRCh38.chr22.dict
11:10:39.871 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:10:39.883 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/0f46kbys4kjgkjkxr66v_d7w0000gn/T/libgkl_compression2050413984379559928.dylib: dlopen(/private/var/folders/sx

## Mark duplicates 

In [14]:
!gatk MarkDuplicates \
  -I ./output2/bwa/S01_sorted.bam \
  -O ./output2/bwa/S01_marked.bam \
  -M ./output2/bwa/S01_marked.metrics.txt

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar MarkDuplicates -I ./output2/bwa/S01_sorted.bam -O ./output2/bwa/S01_marked.bam -M ./output2/bwa/S01_marked.metrics.txt
11:12:02.555 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:12:02.572 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/0f46kbys4kjgkjkxr66v_d7w0000gn/T/libgkl_compression8307784403335199963.dylib: dlopen(/private/var/fo

In [15]:
!samtools index ./output2/bwa/S01_marked.bam

## Variant calling

In [16]:
!mkdir -p ./output2/variants

!gatk HaplotypeCaller \
  -R ./reference_chr22/hg38_GRCh38.chr22.fa \
  -I ./output2/bwa/S01_marked.bam \
  -O ./output2/variants/S01_raw.vcf \
  -L chr22

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R ./reference_chr22/hg38_GRCh38.chr22.fa -I ./output2/bwa/S01_marked.bam -O ./output2/variants/S01_raw.vcf -L chr22
11:13:01.215 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:13:01.222 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/0f46kbys4kjgkjkxr66v_d7w0000gn/T/libgkl_compression11299950897562596674.dylib: dlopen(

## SNP

In [18]:
!gatk SelectVariants \
  -R ./reference_chr22/hg38_GRCh38.chr22.fa \
  -V ./output2/variants/S01_raw.vcf \
  --select-type SNP \
  -O ./output2/variants/S01_snps.vcf

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar SelectVariants -R ./reference_chr22/hg38_GRCh38.chr22.fa -V ./output2/variants/S01_raw.vcf --select-type SNP -O ./output2/variants/S01_snps.vcf
11:31:27.444 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:31:27.452 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/0f46kbys4kjgkjkxr66v_d7w0000gn/T/libgkl_compression3032404412743202967.dyli

In [19]:
!gatk VariantFiltration \
  -R ./reference_chr22/hg38_GRCh38.chr22.fa \
  -V ./output2/variants/S01_snps.vcf \
  -O ./output2/variants/S01_snps_filtered.vcf \
  --filter-name "SNP_filter" \
  --filter-expression "QD < 2.0 || FS > 60.0 || MQ < 40.0"

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar VariantFiltration -R ./reference_chr22/hg38_GRCh38.chr22.fa -V ./output2/variants/S01_snps.vcf -O ./output2/variants/S01_snps_filtered.vcf --filter-name SNP_filter --filter-expression QD < 2.0 || FS > 60.0 || MQ < 40.0
11:31:34.384 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:31:34.391 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/

## INDEL

In [20]:
!gatk SelectVariants \
  -R ./reference_chr22/hg38_GRCh38.chr22.fa \
  -V ./output2/variants/S01_raw.vcf \
  --select-type INDEL \
  -O ./output2/variants/S01_indels.vcf

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar SelectVariants -R ./reference_chr22/hg38_GRCh38.chr22.fa -V ./output2/variants/S01_raw.vcf --select-type INDEL -O ./output2/variants/S01_indels.vcf
11:35:29.739 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:35:29.747 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/0f46kbys4kjgkjkxr66v_d7w0000gn/T/libgkl_compression8647925632435942749.

In [21]:
!gatk VariantFiltration \
  -R ./reference_chr22/hg38_GRCh38.chr22.fa \
  -V ./output2/variants/S01_indels.vcf \
  -O ./output2/variants/S01_indels_filtered.vcf \
  --filter-name "INDEL_filter" \
  --filter-expression "QD < 2.0 || FS > 200.0"

Using GATK jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar VariantFiltration -R ./reference_chr22/hg38_GRCh38.chr22.fa -V ./output2/variants/S01_indels.vcf -O ./output2/variants/S01_indels_filtered.vcf --filter-name INDEL_filter --filter-expression QD < 2.0 || FS > 200.0
11:35:36.228 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/andrinschumacher/miniconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

11:35:36.235 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/sx/0f46kb

## Extraction

In [22]:
!bcftools view -f PASS \
  ./output2/variants/S01_snps_filtered.vcf \
  > ./output2/variants/S01_snps_PASS.vcf

In [23]:
!bcftools view -f PASS \
  ./output2/variants/S01_indels_filtered.vcf \
  > ./output2/variants/S01_indels_PASS.vcf